**Import Key Libraries**

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import lxml
import pandas as pd
import re
from datetime import datetime
from selenium.webdriver.common.keys import Keys 

In [2]:
%pip install selenium

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 2.2 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.1 MB 2.1 MB/s eta 0:00:06
   ----- ---------------------------------- 1.6/12.1 MB 2.2 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.1 MB 2.6 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/12.1 MB 2.8 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.1 MB 3.1 MB/s eta 0:00:03
   ---------------- ----------------------- 5.0/12.1 MB 3.3 MB/s eta 0:00:03
   -------------------- ------------------- 6.3/12.1 MB 3.6 MB/s eta 0:00:02
   -------------------------- ------------- 7.9/12.1 MB 4.1 MB/s eta 0:00:02
   ------------------------------ --------- 9.2/12.1 MB 4.3 MB/s eta 0:00:01
   -------------------

### Step1: Get all product Links

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
import pandas as pd
from datetime import datetime
import time


# ============================================================
# INPUTS
# ============================================================

search_box_text = "sports shoes for women"
website_link = "https://www.flipkart.com/"

TOTAL_PAGES = 25


# ============================================================
# START BROWSER
# ============================================================

session_start_time = datetime.now().time()

print(f"Session Start Time: {session_start_time}")
print("-" * 80)

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

try:

    # ========================================================
    # OPEN FLIPKART
    # ========================================================

    print("Opening Flipkart...")
    driver.get(website_link)

    wait = WebDriverWait(driver, 30)

    # ========================================================
    # SEARCH
    # ========================================================

    print("Waiting for search input...")

    search_input = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'input[autocomplete="off"]')
        )
    )

    print("Typing search query...")

    search_input.clear()
    search_input.send_keys(search_box_text)

    print("Submitting search...")

    search_input.send_keys(Keys.RETURN)

    # Give Flipkart some time to render results
    time.sleep(5)

    # ========================================================
    # WAIT FOR SEARCH RESULTS
    # ========================================================

    print("Waiting for search results...")

    try:
        wait.until(
            lambda d: len(
                d.find_elements(By.CSS_SELECTOR, 'a[href*="/p/"]')
            ) > 0
        )
    except TimeoutException:
        print("WARNING: Product links were not detected on search page.")

    # ========================================================
    # GET FIRST PAGE URL
    # ========================================================

    print("Collecting pagination information...")

    current_url = driver.current_url

    print("Search URL:")
    print(current_url)

    # Flipkart pagination normally uses &page=2, &page=3, etc.
    # Instead of modifying the first pagination link manually,
    # construct the page URLs safely.

    if "page=" in current_url:
        first_page_link = current_url.split("page=")[0] + "page=1"
    else:
        separator = "&" if "?" in current_url else "?"
        first_page_link = current_url + separator + "page=1"

    all_pagination_links = []

    for page_number in range(1, TOTAL_PAGES + 1):

        if "page=" in first_page_link:
            page_link = first_page_link.split("page=")[0] + f"page={page_number}"
        else:
            page_link = first_page_link

        all_pagination_links.append(page_link)

    print("Pagination Links Count:", len(all_pagination_links))

    # ========================================================
    # COLLECT PRODUCT LINKS
    # ========================================================

    print("\nCollecting Product Detail Page Links...")
    print("-" * 80)

    all_product_links = []

    failed_pages = []

    for page_number, link in enumerate(all_pagination_links, start=1):

        print(f"\nProcessing Page {page_number}/{TOTAL_PAGES}")
        print("URL:", link)

        try:

            driver.get(link)

            # Wait for browser document to finish loading
            WebDriverWait(driver, 30).until(
                lambda d: d.execute_script(
                    "return document.readyState"
                ) == "complete"
            )

            # Give JavaScript time to render products
            time.sleep(3)

            # ====================================================
            # FIND PRODUCT LINKS
            # ====================================================

            product_elements = []

            # Primary selector
            selectors = [
                'a[href*="/p/"]',
                'a[href*="/p/"][target="_blank"]',
                'a[href*="/p/"] div'
            ]

            for selector in selectors:

                try:

                    elements = driver.find_elements(
                        By.CSS_SELECTOR,
                        selector
                    )

                    if elements:
                        product_elements.extend(elements)

                except Exception:
                    pass

            # ====================================================
            # EXTRACT HREFS
            # ====================================================

            page_product_links = []

            for element in product_elements:

                try:

                    href = element.get_attribute("href")

                    if href and "/p/" in href:

                        # Remove tracking/query parameters
                        href = href.split("?")[0]

                        page_product_links.append(href)

                except Exception:
                    continue

            # Remove duplicates from current page
            page_product_links = list(
                dict.fromkeys(page_product_links)
            )

            # ====================================================
            # CHECK RESULT
            # ====================================================

            if len(page_product_links) == 0:

                print(
                    f"WARNING: No product links found on page {page_number}"
                )

                failed_pages.append(
                    {
                        "page": page_number,
                        "url": link,
                        "reason": "No product links found"
                    }
                )

                # Save screenshot so you can inspect what Flipkart returned
                try:
                    driver.save_screenshot(
                        f"flipkart_page_{page_number}_failed.png"
                    )
                except Exception:
                    pass

            else:

                print(
                    f"Products found on page {page_number}: "
                    f"{len(page_product_links)}"
                )

                all_product_links.extend(page_product_links)

            # Small delay between pages
            time.sleep(2)

        except TimeoutException:

            print(
                f"TIMEOUT: Page {page_number} could not load properly."
            )

            failed_pages.append(
                {
                    "page": page_number,
                    "url": link,
                    "reason": "Timeout"
                }
            )

        except WebDriverException as e:

            print(
                f"WEBDRIVER ERROR on page {page_number}: {e}"
            )

            failed_pages.append(
                {
                    "page": page_number,
                    "url": link,
                    "reason": "WebDriver error"
                }
            )

        except Exception as e:

            print(
                f"ERROR on page {page_number}: {e}"
            )

            failed_pages.append(
                {
                    "page": page_number,
                    "url": link,
                    "reason": str(e)
                }
            )

    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print("\n" + "=" * 80)
    print("Cleaning Product Links...")
    print("=" * 80)

    all_product_links = list(
        dict.fromkeys(all_product_links)
    )

    print(
        "Total Unique Product Detail Page Links:",
        len(all_product_links)
    )

    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    df_product_links = pd.DataFrame(
        all_product_links,
        columns=["product_links"]
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    output_file = "flipkart_product_links.csv"

    df_product_links.to_csv(
        output_file,
        index=False
    )

    print("\nProduct links saved successfully:")
    print(output_file)

    # ========================================================
    # FAILED PAGE REPORT
    # ========================================================

    if failed_pages:

        failed_df = pd.DataFrame(failed_pages)

        failed_df.to_csv(
            "flipkart_failed_pages.csv",
            index=False
        )

        print("\nWARNING: Some pages failed.")

        print(failed_df)

        print(
            "\nFailed page report saved as:"
            " flipkart_failed_pages.csv"
        )

    else:

        print("\nAll pages processed successfully.")

finally:

    # ========================================================
    # CLOSE BROWSER
    # ========================================================

    try:
        driver.quit()
    except Exception:
        pass

    session_end_time = datetime.now().time()

    print("\n" + "=" * 80)
    print(f"Session End Time: {session_end_time}")
    print("=" * 80)

Session Start Time: 11:23:51.327829
--------------------------------------------------------------------------------
Opening Flipkart...
Waiting for search input...
Typing search query...
Submitting search...
Waiting for search results...
Search URL:
https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off
Pagination Links Count: 50

--------------------------------------------------------------------------------

Processing Page 1/50
URL: https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=1
Products found on page 1: 40

Processing Page 2/50
URL: https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=2
Products found on page 2: 40

Processing Page 3/50
URL: https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&

### Step2: Get Individual product information

In [6]:
import pandas as pd
import re
import time
import json

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException
)


# ============================================================
# SETTINGS
# ============================================================

INPUT_FILE = "flipkart_product_links.csv"

OUTPUT_PRODUCT_FILE = "flipkart_product_data.csv"
OUTPUT_UNAVAILABLE_FILE = "unavailable_products.csv"
OUTPUT_DUPLICATE_FILE = "duplicate_products.csv"
OUTPUT_FAILED_FILE = "failed_product_urls.csv"

# ------------------------------------------------------------
# Set this to:
#
# 10    -> scrape only 10 products for testing
# 100   -> scrape 100 products
# None  -> scrape ALL products from CSV
# ------------------------------------------------------------

MAX_PRODUCTS = None


# ============================================================
# SESSION START
# ============================================================

session_start_time = datetime.now().time()

print(
    f"Session Start Time: {session_start_time}"
    " ---------------------------> "
)

print("=" * 80)


# ============================================================
# READ PRODUCT LINKS
# ============================================================

print("Reading product links CSV...")

df_product_links = pd.read_csv(INPUT_FILE)

if "product_links" not in df_product_links.columns:
    raise ValueError(
        "CSV does not contain a 'product_links' column."
    )


# Remove empty links
df_product_links = df_product_links.dropna(
    subset=["product_links"]
)


# Remove duplicate URLs
df_product_links = df_product_links.drop_duplicates(
    subset=["product_links"]
)


# ============================================================
# LIMIT PRODUCTS FOR TESTING
# ============================================================

if MAX_PRODUCTS is not None:

    df_product_links = df_product_links.head(
        MAX_PRODUCTS
    )


all_product_links = (
    df_product_links["product_links"]
    .astype(str)
    .tolist()
)


print(
    f"Product URLs selected for scraping: "
    f"{len(all_product_links)}"
)

print("=" * 80)


# ============================================================
# START BROWSER
# ============================================================

print("Starting Chrome browser...")

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 30)


# ============================================================
# JSON-LD HELPER FUNCTIONS
# ============================================================

def find_product_jsonld(data):
    """
    Recursively search JSON-LD data for a Product object.

    Handles:
    - Dictionary
    - List
    - @graph
    - Multiple JSON-LD objects
    """

    if isinstance(data, dict):

        # Direct Product object
        product_type = data.get("@type")

        if isinstance(product_type, str):
            if product_type.lower() == "product":
                return data

        elif isinstance(product_type, list):
            if any(
                str(item).lower() == "product"
                for item in product_type
            ):
                return data

        # Search inside @graph
        if "@graph" in data:

            result = find_product_jsonld(
                data["@graph"]
            )

            if result:
                return result

        # Search all nested values
        for value in data.values():

            result = find_product_jsonld(value)

            if result:
                return result

    elif isinstance(data, list):

        for item in data:

            result = find_product_jsonld(item)

            if result:
                return result

    return None


def get_jsonld_product(driver):
    """
    Extract the Product JSON-LD object from the page.
    """

    scripts = driver.find_elements(
        By.CSS_SELECTOR,
        'script[type="application/ld+json"]'
    )

    for script in scripts:

        try:

            raw_json = script.get_attribute(
                "textContent"
            )

            if not raw_json:
                continue

            raw_json = raw_json.strip()

            if not raw_json:
                continue

            data = json.loads(raw_json)

            product = find_product_jsonld(data)

            if product:
                return product

        except (
            json.JSONDecodeError,
            TypeError,
            ValueError
        ):

            # Some websites contain malformed JSON-LD.
            # Ignore that script and check the next one.
            continue

    return None


def get_brand(product_data):
    """
    Extract brand from JSON-LD.
    """

    brand = product_data.get("brand", "")

    if isinstance(brand, dict):

        return str(
            brand.get("name", "")
        ).strip()

    if isinstance(brand, str):

        return brand.strip()

    return ""


def get_offer_data(product_data):
    """
    Extract offer information from JSON-LD.

    Handles:
    - offers as dictionary
    - offers as list
    """

    offers = product_data.get(
        "offers",
        {}
    )

    if isinstance(offers, dict):

        return offers

    if isinstance(offers, list):

        for offer in offers:

            if isinstance(offer, dict):
                return offer

    return {}


def get_discount_from_jsonld(product_data, offers):
    """
    Extract discount only if JSON-LD explicitly provides it.

    We do NOT calculate a fake discount from unrelated
    price values because JSON-LD may not contain MRP.
    """

    # --------------------------------------------------------
    # Direct discount fields
    # --------------------------------------------------------

    possible_discount_fields = [
        "discount",
        "discountPercentage",
        "discount_percent",
        "discountPercent"
    ]

    for field in possible_discount_fields:

        value = product_data.get(field)

        if value not in (None, ""):

            value = str(value).strip()

            numbers = re.findall(
                r"\d+(?:\.\d+)?",
                value
            )

            if numbers:

                number = float(numbers[0])

                # Keep the same format as your old scraper:
                # 20% -> 0.2
                if number > 1:
                    return number / 100

                return number

    # --------------------------------------------------------
    # Check offers for discount
    # --------------------------------------------------------

    for field in possible_discount_fields:

        value = offers.get(field)

        if value not in (None, ""):

            value = str(value).strip()

            numbers = re.findall(
                r"\d+(?:\.\d+)?",
                value
            )

            if numbers:

                number = float(numbers[0])

                if number > 1:
                    return number / 100

                return number

    # JSON-LD doesn't explicitly provide discount
    return ""


def extract_product_attributes(driver, product_page_link):
    """
    Extract product attributes ONLY from JSON-LD.

    Returns:
        product_link
        title
        brand
        price
        discount
        avg_rating
        total_ratings
    """

    product_data = get_jsonld_product(driver)

    if not product_data:

        print(
            "WARNING: Product JSON-LD could not be found."
        )

        return None

    # ========================================================
    # PRODUCT LINK
    # ========================================================

    product_link = product_page_link


    # ========================================================
    # TITLE
    # ========================================================

    title = product_data.get(
        "name",
        ""
    )

    if title is None:
        title = ""

    title = str(title).strip()

    # Keep your existing title cleanup
    if title:

        title = re.sub(
            r"\s*\([^)]*\)",
            "",
            title
        ).strip()


    # ========================================================
    # BRAND
    # ========================================================

    brand = get_brand(
        product_data
    )


    # ========================================================
    # OFFERS
    # ========================================================

    offers = get_offer_data(
        product_data
    )


    # ========================================================
    # PRICE
    # ========================================================

    price = offers.get(
        "price",
        ""
    )

    if price is None:
        price = ""

    price = str(price).strip()

    # Keep numeric price only
    if price:

        price_numbers = re.findall(
            r"\d+(?:\.\d+)?",
            price
        )

        if price_numbers:

            price = price_numbers[0]


    # ========================================================
    # DISCOUNT
    # ========================================================

    discount = get_discount_from_jsonld(
        product_data,
        offers
    )


    # ========================================================
    # AGGREGATE RATING
    # ========================================================

    aggregate_rating = product_data.get(
        "aggregateRating",
        {}
    )

    if not isinstance(
        aggregate_rating,
        dict
    ):

        aggregate_rating = {}


    # ========================================================
    # AVERAGE RATING
    # ========================================================

    avg_rating = aggregate_rating.get(
        "ratingValue",
        ""
    )

    if avg_rating is None:
        avg_rating = ""

    avg_rating = str(
        avg_rating
    ).strip()


    # ========================================================
    # TOTAL RATINGS
    # ========================================================

    # Prefer ratingCount because this is the total
    # number of ratings.
    total_ratings = aggregate_rating.get(
        "ratingCount",
        ""
    )

    # Fallback to reviewCount if ratingCount
    # isn't available.
    if total_ratings in (None, ""):

        total_ratings = aggregate_rating.get(
            "reviewCount",
            ""
        )

    if total_ratings is None:
        total_ratings = ""

    total_ratings = str(
        total_ratings
    ).strip()

    if total_ratings:

        rating_numbers = re.findall(
            r"\d[\d,]*",
            total_ratings
        )

        if rating_numbers:

            total_ratings = int(
                rating_numbers[0].replace(
                    ",",
                    ""
                )
            )


    # ========================================================
    # PRINT EXTRACTED DATA
    # ========================================================

    print("\nJSON-LD PRODUCT DATA FOUND")

    print(
        f"Title: {title}"
    )

    print(
        f"Brand: {brand}"
    )

    print(
        f"Price: {price}"
    )

    print(
        f"Discount: {discount}"
    )

    print(
        f"Average Rating: {avg_rating}"
    )

    print(
        f"Total Ratings: {total_ratings}"
    )


    # ========================================================
    # RETURN
    # ========================================================

    return [
        product_link,
        title,
        brand,
        price,
        discount,
        avg_rating,
        total_ratings
    ]


# ============================================================
# STORAGE
# ============================================================

complete_product_details = []

unavailable_products = []

failed_product_urls = []

duplicate_product_details = []

successful_parsed_urls_count = 0

complete_failed_urls_count = 0

unavailable_products_count = 0


# ============================================================
# PRODUCT SCRAPING
# ============================================================

print(
    "Collecting Individual Product Detail Information"
)

print("=" * 80)


for index, product_page_link in enumerate(
    all_product_links,
    start=1
):

    print("\n" + "-" * 80)

    print(
        f"Processing Product "
        f"{index}/{len(all_product_links)}"
    )

    print(
        f"URL: {product_page_link}"
    )

    try:

        # ====================================================
        # OPEN PRODUCT PAGE
        # ====================================================

        driver.get(
            product_page_link
        )

        # Wait for browser document to finish loading
        WebDriverWait(driver, 30).until(
            lambda d: d.execute_script(
                "return document.readyState"
            ) == "complete"
        )

        # Give JavaScript time to render
        time.sleep(2)


        # ====================================================
        # CHECK FOR UNAVAILABLE PRODUCT
        # ====================================================

        page_text = driver.find_element(
            By.TAG_NAME,
            "body"
        ).text.lower()


        unavailable_keywords = [
            "currently unavailable",
            "sold out",
            "out of stock"
        ]


        is_unavailable = any(
            keyword in page_text
            for keyword in unavailable_keywords
        )


        if is_unavailable:

            print(
                "Product is unavailable."
            )

            unavailable_products.append(
                product_page_link
            )

            unavailable_products_count += 1

            continue


        # ====================================================
        # EXTRACT ALL PRODUCT ATTRIBUTES
        # FROM JSON-LD
        # ====================================================

        product_details = extract_product_attributes(
            driver,
            product_page_link
        )


        # ====================================================
        # VALIDATE BASIC PRODUCT DATA
        # ====================================================

        if not product_details:

            print(
                "WARNING: Product JSON-LD "
                "could not be extracted."
            )

            failed_product_urls.append(
                {
                    "product_link": product_page_link,
                    "reason": "Product JSON-LD not found"
                }
            )

            complete_failed_urls_count += 1

            continue


        # Unpack data
        (
            product_link,
            title,
            brand,
            price,
            discount,
            avg_rating,
            total_ratings
        ) = product_details


        # Title and price are still required
        if not title or not price:

            print(
                "WARNING: Product title or price "
                "could not be extracted from JSON-LD."
            )

            failed_product_urls.append(
                {
                    "product_link": product_page_link,
                    "reason": "Title or price not found in JSON-LD"
                }
            )

            complete_failed_urls_count += 1

            continue


        # ====================================================
        # SAVE PRODUCT
        # ====================================================

        complete_product_details.append(
            [
                product_link,
                title,
                brand,
                price,
                discount,
                avg_rating,
                total_ratings
            ]
        )

        successful_parsed_urls_count += 1

        print(
            f"SUCCESS: Product "
            f"{successful_parsed_urls_count} parsed."
        )


    except TimeoutException as e:

        print(
            f"TIMEOUT while processing:"
            f"\n{product_page_link}"
        )

        failed_product_urls.append(
            {
                "product_link": product_page_link,
                "reason": "Page load timeout"
            }
        )

        complete_failed_urls_count += 1


    except WebDriverException as e:

        print(
            f"WEBDRIVER ERROR:"
            f"\n{product_page_link}"
        )

        failed_product_urls.append(
            {
                "product_link": product_page_link,
                "reason": "WebDriver error"
            }
        )

        complete_failed_urls_count += 1


    except Exception as e:

        print(
            f"FAILED:"
            f"\n{product_page_link}"
            f"\nReason: {e}"
        )

        failed_product_urls.append(
            {
                "product_link": product_page_link,
                "reason": str(e)
            }
        )

        complete_failed_urls_count += 1


    # Small delay between products
    time.sleep(1)


# ============================================================
# CREATE PRODUCT DATAFRAME
# ============================================================

print("\n" + "=" * 80)

print(
    "Creating product DataFrame..."
)

print("=" * 80)


df = pd.DataFrame(
    complete_product_details,
    columns=[
        "product_link",
        "title",
        "brand",
        "price",
        "discount",
        "avg_rating",
        "total_ratings"
    ]
)


# ============================================================
# DUPLICATE PRODUCTS
# ============================================================

if not df.empty:

    duplicate_mask = df.duplicated(
        subset=[
            "brand",
            "price",
            "discount",
            "avg_rating",
            "total_ratings"
        ],
        keep="first"
    )

    df_duplicate_products = df[
        duplicate_mask
    ].copy()

    df = df[
        ~duplicate_mask
    ].copy()

else:

    df_duplicate_products = pd.DataFrame(
        columns=[
            "product_link",
            "title",
            "brand",
            "price",
            "discount",
            "avg_rating",
            "total_ratings"
        ]
    )


# ============================================================
# UNAVAILABLE PRODUCTS
# ============================================================

df_unavailable_products = pd.DataFrame(
    unavailable_products,
    columns=["link"]
)


# ============================================================
# FAILED PRODUCTS
# ============================================================

df_failed_products = pd.DataFrame(
    failed_product_urls,
    columns=[
        "product_link",
        "reason"
    ]
)


# ============================================================
# PRINT FINAL STATISTICS
# ============================================================

print("\n" + "=" * 80)

print(
    "SCRAPING SUMMARY"
)

print("=" * 80)


print(
    "Total product URLs selected: ",
    len(all_product_links)
)


print(
    "Successfully parsed products: ",
    successful_parsed_urls_count
)


print(
    "Failed product URLs: ",
    complete_failed_urls_count
)


print(
    "Unavailable products: ",
    len(df_unavailable_products)
)


print(
    "Duplicate products removed: ",
    len(df_duplicate_products)
)


print(
    "Final unique products: ",
    len(df)
)


print("=" * 80)


# ============================================================
# SAVE FILES
# ============================================================

print(
    "\nSaving files..."
)


df.to_csv(
    OUTPUT_PRODUCT_FILE,
    index=False
)


df_unavailable_products.to_csv(
    OUTPUT_UNAVAILABLE_FILE,
    index=False
)


df_duplicate_products.to_csv(
    OUTPUT_DUPLICATE_FILE,
    index=False
)


df_failed_products.to_csv(
    OUTPUT_FAILED_FILE,
    index=False
)


print(
    f"Product data saved: "
    f"{OUTPUT_PRODUCT_FILE}"
)


print(
    f"Unavailable products saved: "
    f"{OUTPUT_UNAVAILABLE_FILE}"
)


print(
    f"Duplicate products saved: "
    f"{OUTPUT_DUPLICATE_FILE}"
)


print(
    f"Failed URLs saved: "
    f"{OUTPUT_FAILED_FILE}"
)


# ============================================================
# CLOSE BROWSER
# ============================================================

try:

    driver.quit()

except Exception:

    pass


# ============================================================
# SESSION END
# ============================================================

session_end_time = datetime.now().time()

print(
    "\n" + "=" * 80
)

print(
    f"Session End Time: "
    f"{session_end_time}"
)

print("=" * 80)

Session Start Time: 11:35:19.166570 ---------------------------> 
Reading product links CSV...
Product URLs selected for scraping: 589
Starting Chrome browser...

--------------------------------------------------------------------------------
Processing Product 1/589
URL: https://www.flipkart.com/adidas-duramo-sl-w-running-shoes-women/p/itm61f4a996ffdef
Product is unavailable.

--------------------------------------------------------------------------------
Processing Product 2/589
URL: https://www.flipkart.com/under-armour-ascend-running-shoes-women/p/itm4be7abc174a48
Product is unavailable.

--------------------------------------------------------------------------------
Processing Product 3/589
URL: https://www.flipkart.com/asian-mercury-11-mint-sports-gym-walking-training-stylish-running-shoes-women/p/itmd59c8ace18690

JSON-LD PRODUCT DATA FOUND
Title: asian Mercury-11 Mint Sports,Gym,Walking,Training,Stylish Running Shoes For Women
Brand: asian
Price: 805
Discount: 
Average Ratin

KeyboardInterrupt: 